In [ ]:
!pip install -q diffusers transformers accelerate opencv-python pillow scikit-image lpips

In [ ]:
!nvidia-smi

Sun Apr 19 22:07:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os

path = project_path + "/src/models/inference.py"

with open(path, "r") as f:
    code = f.read()

new_class = """
class SDConditionControlNet(Pipeline):
    name = "sd_condition_controlnet"

    def __init__(self, device: str | None = None):
        from diffusers import (
            StableDiffusionControlNetPipeline, ControlNetModel,
            UniPCMultistepScheduler,
        )

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if self.device == "cuda" else torch.float32

        controlnet = ControlNetModel.from_pretrained(
            config.CONTROLNET_DEPTH_ID, torch_dtype=dtype,
        )

        self.pipe = StableDiffusionControlNetPipeline.from_pretrained(
            config.SD_MODEL_ID,
            controlnet=controlnet,
            torch_dtype=dtype,
            safety_checker=None,
        ).to(self.device)

        self.pipe.scheduler = UniPCMultistepScheduler.from_config(self.pipe.scheduler.config)

    def generate(self, source, location, time_of_day, weather, seed=42):
        src = source.convert("RGB").resize((config.IMAGE_SIZE, config.IMAGE_SIZE))

        prompt = f"A realistic photo of {location} on a university campus, at {time_of_day}, with {weather} weather, natural lighting, preserve structure"

        negative_prompt = "distorted, unrealistic, blurry"

        depth = estimate_depth(src)

        g = torch.Generator(self.device).manual_seed(seed)

        out = self.pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=depth,
            num_inference_steps=config.NUM_INFERENCE_STEPS,
            guidance_scale=config.GUIDANCE_SCALE,
            controlnet_conditioning_scale=config.CONTROLNET_CONDITIONING_SCALE,
            generator=g,
        ).images[0]

        return InferenceResult(image=out, depth=depth)
"""

if "SDConditionControlNet" not in code:
    code += new_class


code = code.replace(
    "raise ValueError(f\"Unknown system: {system}\")",
    """
    if system == "sd_condition_controlnet":
        return SDConditionControlNet()

    raise ValueError(f"Unknown system: {system}")
    """
)

with open(path, "w") as f:
    f.write(code)

print(" inference.py updated")

 inference.py updated


In [ ]:
path = project_path + "/src/eval/ablate.py"

with open(path, "r") as f:
    code = f.read()

code = code.replace(
    'choices=["ip2p", "sd_cn", "sd_cn_lora", "sd_lora"]',
    'choices=["ip2p", "sd_cn", "sd_cn_lora", "sd_lora", "sd_condition_controlnet"]'
)

code = code.replace(
    '"sd_lora": "sd_lora_img2img"}',
    '"sd_lora": "sd_lora_img2img", "sd_condition_controlnet": "sd_condition_controlnet"}'
)

with open(path, "w") as f:
    f.write(code)

print("ablate.py updated")

ablate.py updated


In [ ]:
!pip install -q open_clip_torch

In [ ]:
!python -m src.eval.ablate --systems sd_condition_controlnet --limit 3

[ablate] 56 val pairs, running systems: ['sd_condition_controlnet']
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading pipeline components...:  33% 2/6 [00:08<00:14,  3.65s/it]
Loading weights:   0% 0/196 [00:00<?, ?it/s]
Loading weights:   1% 1/196 [00:00<00:00, 5882.61it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weights:   1% 1/196 [00:00<00:00, 2796.20it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weigh

In [1]:
!python -m src.eval.ablate --systems sd_condition_controlnet

/usr/bin/python3: Error while finding module specification for 'src.eval.ablate' (ModuleNotFoundError: No module named 'src')
